# 08 — Nessie Catalog Refs (catalog-level)

Nessie versions the **entire catalog** as a Git-like commit graph. Every catalog
mutation (create_table, append, overwrite, drop, schema change) is a commit on a
branch (default `main`). You can:

- list the commit history,
- read any past version by pointing a fresh `RestCatalog` at a commit hash,
- create branches and tags via the Nessie API,
- write on a branch without affecting `main`.

This is the real "time travel" pattern for this stack. Iceberg's own `snapshot_id`-based
time travel doesn't work on Nessie because Nessie's `metadata.json` retains only the
current snapshot — history lives at the catalog layer.

**Prerequisites:** Run `01_write_iceberg_polars.ipynb` (more runs of 02–05 give a richer commit log).

In [1]:
import os

import polars as pl
import requests
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.io.pyarrow import schema_to_pyarrow

NESSIE_URI       = os.environ["NESSIE_URI"]              # http://nessie:19120/iceberg
NESSIE_API       = NESSIE_URI.replace("/iceberg", "/api/v2")
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

S3_PROPS = {
    "warehouse": WAREHOUSE_URI,
    "s3.endpoint": S3_ENDPOINT,
    "s3.access-key-id": S3_ACCESS_KEY,
    "s3.secret-access-key": S3_SECRET_KEY,
    "s3.path-style-access": "true",
    "s3.region": "us-east-1",
}

print("Nessie Iceberg REST :", NESSIE_URI)
print("Nessie native API   :", NESSIE_API)

Nessie Iceberg REST : http://nessie:19120/iceberg
Nessie native API   : http://nessie:19120/api/v2


## 1. List the commit history of `main`

Each commit corresponds to one catalog mutation (create_table, append, etc).

In [2]:
r = requests.get(f"{NESSIE_API}/trees/main/history", params={"maxRecords": 20})
r.raise_for_status()
entries = r.json()["logEntries"]
print(f"Recent commits on main ({len(entries)}):")
for e in entries[:10]:
    cm = e["commitMeta"]
    print(f"  {cm['hash'][:12]}  {cm['commitTime']}  {cm.get('message','')[:60]}")

Recent commits on main (16):
  13e3024a6672  2026-05-15T11:06:43.542344762Z  Update ICEBERG_TABLE demo.events
  a6cde488a42b  2026-05-15T11:06:43.499709554Z  Update ICEBERG_TABLE demo.events
  b25ca7e87c7b  2026-05-15T11:06:37.268068551Z  Update ICEBERG_TABLE demo.events
  37656d988880  2026-05-15T11:06:37.016671301Z  Update ICEBERG_TABLE demo.events
  7e1e7cb73f06  2026-05-15T11:06:36.803059676Z  Update ICEBERG_TABLE demo.events
  1436dd883324  2026-05-15T11:06:33.042648674Z  Update ICEBERG_TABLE demo.events
  f1616aed6ef4  2026-05-15T11:06:32.858100882Z  Update ICEBERG_TABLE demo.events
  8aed8041c469  2026-05-15T11:06:29.661252631Z  Update ICEBERG_TABLE demo.events
  faff5446419e  2026-05-15T11:06:29.624245797Z  Update ICEBERG_TABLE demo.events
  206eecca6ff7  2026-05-15T11:06:29.562092172Z  Update ICEBERG_TABLE demo.events


## 2. Read a past version by commit hash

Point a fresh `RestCatalog` at `{NESSIE_URI}/main@{hash}` — Nessie's URL grammar
for "a named ref at a specific commit". A bare hash without a ref name doesn't
work because Nessie's parser requires ref names to start with a letter.

In [3]:
# Pick a hash a few commits back so the table state differs from main HEAD.
# Going too far back lands on a commit before the table existed.
past_idx  = min(5, len(entries) - 1)
past_hash = entries[past_idx]["commitMeta"]["hash"]
print(f"Reading at commit #{past_idx}: {past_hash[:12]}")

catalog_past = RestCatalog(
    name="nessie-past",
    **{**S3_PROPS, "uri": f"{NESSIE_URI}/main@{past_hash}"},
)

try:
    t_past = catalog_past.load_table(("demo", "events"))
    df_past = pl.from_arrow(t_past.scan().to_arrow())
    print(f"Rows at past commit: {len(df_past)}")
    print(f"Rows at main HEAD:   {len(pl.from_arrow(RestCatalog(name='m', **{**S3_PROPS, 'uri': NESSIE_URI}).load_table(('demo','events')).scan().to_arrow()))}")
except Exception as e:
    print("Table didn't exist at that commit yet:", e)

Reading at commit #5: 1436dd883324
Rows at past commit: 10


Rows at main HEAD:   11


## 3. Create a Nessie branch `dev` from `main`

Nessie API v2 ref creation: `POST /trees?name=<new>&type=branch|tag` with the **source** reference
(`{type, name, hash}`) in the JSON body. Delete uses `DELETE /trees/{name}@{hash}` —
the expected hash is embedded in the URL with `@`.

In [4]:
# Need current main hash for the source ref.
main_ref = requests.get(f"{NESSIE_API}/trees/main").json()["reference"]
main_hash = main_ref["hash"]

# Idempotent: if `dev` already exists, delete it first.
existing = requests.get(f"{NESSIE_API}/trees/dev")
if existing.status_code == 200:
    h = existing.json()["reference"]["hash"]
    requests.delete(f"{NESSIE_API}/trees/dev@{h}")

create = requests.post(
    f"{NESSIE_API}/trees",
    params={"name": "dev", "type": "branch"},
    json={"type": "BRANCH", "name": "main", "hash": main_hash},
)
create.raise_for_status()
print("Created branch dev →", create.json()["reference"]["hash"][:12])

Created branch dev → 13e3024a6672


## 4. Write on `dev` — `main` is unaffected

In [5]:
import datetime

catalog_dev = RestCatalog(
    name="nessie-dev",
    **{**S3_PROPS, "uri": f"{NESSIE_URI}/dev"},
)
t_dev = catalog_dev.load_table(("demo", "events"))

UTC = datetime.timezone.utc
kind_col = "event_kind" if "event_kind" in {f.name for f in t_dev.schema().fields} else "event_type"

dev_batch = pl.DataFrame(
    {
        "event_id":   ["e_dev_1"],
        "user_id":    [999],
        kind_col:      ["click"],
        "amount":     [0.0],
        "ts":         [datetime.datetime(2024, 4, 1, 0, 0, 0, tzinfo=UTC)],
    }
)

field_names = [f.name for f in t_dev.schema().fields]
t_dev.append(
    dev_batch.select(field_names).to_arrow().cast(schema_to_pyarrow(t_dev.schema()))
)
t_dev.refresh()

catalog_main = RestCatalog(name="nessie-main", **{**S3_PROPS, "uri": NESSIE_URI})
t_main = catalog_main.load_table(("demo", "events"))

rows_dev  = len(pl.from_arrow(t_dev.scan().to_arrow()))
rows_main = len(pl.from_arrow(t_main.scan().to_arrow()))
print(f"rows on dev:  {rows_dev}")
print(f"rows on main: {rows_main}  (unchanged)")

rows on dev:  12
rows on main: 11  (unchanged)


## 5. Create a Nessie tag pinning the current main

In [6]:
existing = requests.get(f"{NESSIE_API}/trees/release-v1")
if existing.status_code == 200:
    h = existing.json()["reference"]["hash"]
    requests.delete(f"{NESSIE_API}/trees/release-v1@{h}")

tag = requests.post(
    f"{NESSIE_API}/trees",
    params={"name": "release-v1", "type": "tag"},
    json={"type": "BRANCH", "name": "main", "hash": main_hash},
)
tag.raise_for_status()
print("Tag release-v1 →", tag.json()["reference"]["hash"][:12])

Tag release-v1 → 13e3024a6672


## 6. Diff `main` vs `dev`

In [7]:
diff = requests.get(f"{NESSIE_API}/trees/main/diff/dev").json()
print("Diffs:", len(diff.get("diffs", [])))
for d in diff.get("diffs", [])[:5]:
    print(" ", d["key"], "from", d.get("from"), "→", d.get("to"))

Diffs: 1
  {'elements': ['demo', 'events']} from {'type': 'ICEBERG_TABLE', 'id': '76fad100-fdc8-4bcf-ab39-a3c687a2ce40', 'metadataLocation': 's3://iceberg-warehouse/warehouse/demo/events_61443c3e-6418-4c05-93a5-bcbe3b99cd97/metadata/00000-3bbe4294-90db-48b1-962c-138aee11f718.metadata.json', 'snapshotId': 6746404416189507434, 'schemaId': 4, 'specId': 1, 'sortOrderId': 0} → {'type': 'ICEBERG_TABLE', 'id': '76fad100-fdc8-4bcf-ab39-a3c687a2ce40', 'metadataLocation': 's3://iceberg-warehouse/warehouse/demo/events_61443c3e-6418-4c05-93a5-bcbe3b99cd97/metadata/00000-ad45c678-11f5-4b38-b3a5-b5576775fa3c.metadata.json', 'snapshotId': 4267947191827055920, 'schemaId': 4, 'specId': 1, 'sortOrderId': 0}


## 7. When to use which

- **Iceberg table refs** (notebook 07) — version one table independently. Good for tagging a release of *one* dataset, or A/B testing a single table.
- **Nessie catalog refs** (this notebook) — version many tables together. Good for cross-table atomic changes (ETL that touches 5 tables and either all succeed or all roll back), and for the Git-style PR workflow (commit on a branch, review, merge).

---
**Next:** [09_snapshot_management.ipynb](09_snapshot_management.ipynb) — rollback and expire snapshots.